# 02 — Silver: Source Profiling

Profiles the bronze copy of `samples.tpch` **before** we design the silver layer.
Every data-quality rule in `02_silver` (types, NOT NULL, CHECK ranges, allowed values, keys) is derived from what this notebook shows.

What we profile:

1. Schema and row counts
2. NULLs and blank strings per column
3. Key uniqueness (candidate primary keys)
4. Referential integrity — orphans, including the composite `lineitem → partsupp (partkey, suppkey)` key
5. Numeric ranges — quantity, prices, **discount, tax**
6. Categorical domains — the allowed-value lists
7. Date ranges
8. Derived DQ bounds (used by `02_silver`)

Prerequisite: run `01_bronze` first. This notebook is read-only — it creates no tables.

In [0]:
# Parameters (override from a Job or the widget bar to run in another workspace)
dbutils.widgets.text("catalog", "group1_finance", "Catalog")
dbutils.widgets.text("bronze_schema", "bronze", "Bronze schema")

CATALOG = dbutils.widgets.get("catalog")
BRONZE  = f"{CATALOG}.{dbutils.widgets.get('bronze_schema')}"

TPCH_TABLES = [
    "customer",
    "lineitem",
    "nation",
    "orders",
    "part",
    "partsupp",
    "region",
    "supplier",
]

from pyspark.sql import functions as F

def bronze(table):
    return spark.table(f"{BRONZE}.{table}")

print(f"Profiling: {BRONZE}")

## 1. Schema and row counts

In [0]:
rows = [(t, bronze(t).count(), len(bronze(t).columns)) for t in TPCH_TABLES]
display(spark.createDataFrame(rows, "table STRING, row_count LONG, column_count INT"))

In [0]:
schema_rows = [
    (t, i + 1, col, dtype)
    for t in TPCH_TABLES
    for i, (col, dtype) in enumerate(bronze(t).dtypes)
]
display(spark.createDataFrame(schema_rows, "table STRING, position INT, column STRING, data_type STRING"))

## 2. NULLs and blank strings per column

A column is a NOT NULL candidate only if it has zero NULLs here. Blank (`''` after trim) strings are counted separately, because a blank brand or segment is as useless as a NULL one.

In [0]:
null_rows = []
for t in TPCH_TABLES:
    df = bronze(t)
    string_cols = {c for c, d in df.dtypes if d == "string"}
    aggs = []
    for c in df.columns:
        aggs.append(F.sum(F.col(c).isNull().cast("long")).alias(f"{c}__null"))
        if c in string_cols:
            aggs.append(F.sum((F.trim(F.col(c)) == "").cast("long")).alias(f"{c}__blank"))
    r = df.agg(*aggs).first().asDict()
    for c in df.columns:
        null_rows.append((t, c, r[f"{c}__null"] or 0, r.get(f"{c}__blank") or 0))

nulls_df = spark.createDataFrame(null_rows, "table STRING, column STRING, nulls LONG, blanks LONG")
display(nulls_df.orderBy(F.desc("nulls"), F.desc("blanks"), "table", "column"))

problem_cols = nulls_df.filter("nulls > 0 OR blanks > 0").count()
print(f"Columns with NULLs or blanks: {problem_cols}")

## 3. Key uniqueness

Candidate primary keys from the TPC-H spec. `partsupp` and `lineitem` have **composite** keys.
We also count exact duplicate rows (the whole row repeated), which would be a load artefact rather than a business conflict.

In [0]:
from functools import reduce

PRIMARY_KEYS = {
    "region":   ["r_regionkey"],
    "nation":   ["n_nationkey"],
    "supplier": ["s_suppkey"],
    "customer": ["c_custkey"],
    "part":     ["p_partkey"],
    "partsupp": ["ps_partkey", "ps_suppkey"],
    "orders":   ["o_orderkey"],
    "lineitem": ["l_orderkey", "l_linenumber"],
}

key_rows = []
for t, keys in PRIMARY_KEYS.items():
    df = bronze(t)
    r = df.agg(
        F.count("*").alias("rows"),
        F.countDistinct(*keys).alias("distinct_keys"),
        F.sum(reduce(lambda a, b: a | b, [F.col(k).isNull() for k in keys]).cast("long")).alias("null_keys"),
    ).first()
    exact_dupes = r["rows"] - df.distinct().count()
    key_rows.append((t, ", ".join(keys), r["rows"], r["distinct_keys"],
                     r["rows"] - r["distinct_keys"], r["null_keys"] or 0, exact_dupes))

display(spark.createDataFrame(
    key_rows,
    "table STRING, key STRING, rows LONG, distinct_keys LONG, duplicate_keys LONG, null_keys LONG, exact_duplicate_rows LONG",
))

## 4. Referential integrity

Orphan = child row whose foreign key has no matching parent. Note the **two-column** key from `lineitem` to `partsupp`:
a line item must reference a (part, supplier) pair that actually exists in `partsupp`, not just a valid part and a valid supplier separately.

We also count parents without children — those are *not* errors, but they matter for joins:
customers without orders must survive into silver (they are real customers), while an order with zero line items would break order-total reconciliation.

In [0]:
FOREIGN_KEYS = [
    # (child, child_cols, parent, parent_cols)
    ("nation",   ["n_regionkey"],              "region",   ["r_regionkey"]),
    ("supplier", ["s_nationkey"],              "nation",   ["n_nationkey"]),
    ("customer", ["c_nationkey"],              "nation",   ["n_nationkey"]),
    ("partsupp", ["ps_partkey"],               "part",     ["p_partkey"]),
    ("partsupp", ["ps_suppkey"],               "supplier", ["s_suppkey"]),
    ("orders",   ["o_custkey"],                "customer", ["c_custkey"]),
    ("lineitem", ["l_orderkey"],               "orders",   ["o_orderkey"]),
    ("lineitem", ["l_partkey"],                "part",     ["p_partkey"]),
    ("lineitem", ["l_suppkey"],                "supplier", ["s_suppkey"]),
    ("lineitem", ["l_partkey", "l_suppkey"],   "partsupp", ["ps_partkey", "ps_suppkey"]),
]

def orphan_count(child, child_cols, parent, parent_cols):
    parent_keys = bronze(parent).select(*[F.col(p).alias(c) for c, p in zip(child_cols, parent_cols)]).distinct()
    return bronze(child).join(parent_keys, on=child_cols, how="left_anti").count()

fk_rows = [
    (child, ", ".join(cc), parent, ", ".join(pc), orphan_count(child, cc, parent, pc))
    for child, cc, parent, pc in FOREIGN_KEYS
]
display(spark.createDataFrame(fk_rows, "child STRING, child_cols STRING, parent STRING, parent_cols STRING, orphans LONG"))

In [0]:
customers_without_orders = bronze("customer").join(
    bronze("orders").select(F.col("o_custkey").alias("c_custkey")).distinct(), "c_custkey", "left_anti"
).count()
orders_without_lines = bronze("orders").join(
    bronze("lineitem").select(F.col("l_orderkey").alias("o_orderkey")).distinct(), "o_orderkey", "left_anti"
).count()
partsupp_never_ordered = bronze("partsupp").join(
    bronze("lineitem").select(F.col("l_partkey").alias("ps_partkey"), F.col("l_suppkey").alias("ps_suppkey")).distinct(),
    ["ps_partkey", "ps_suppkey"], "left_anti",
).count()

print(f"Customers with no orders      : {customers_without_orders:>10,}  (valid - keep them in silver)")
print(f"Orders with zero line items   : {orders_without_lines:>10,}  (would break order-total reconciliation)")
print(f"partsupp pairs never ordered  : {partsupp_never_ordered:>10,}  (valid - catalogue offers)")

## 5. Numeric ranges

`non_positive` counts values `<= 0`; `negative` counts values `< 0`.
Account balances are profiled too, to decide whether they need a positivity rule (TPC-H allows negative balances).

In [0]:
NUMERIC_COLUMNS = {
    "lineitem": ["l_linenumber", "l_quantity", "l_extendedprice", "l_discount", "l_tax"],
    "orders":   ["o_totalprice", "o_shippriority"],
    "part":     ["p_retailprice", "p_size"],
    "partsupp": ["ps_availqty", "ps_supplycost"],
    "customer": ["c_acctbal"],
    "supplier": ["s_acctbal"],
}

num_rows = []
for t, cols in NUMERIC_COLUMNS.items():
    aggs = []
    for c in cols:
        aggs += [
            F.min(c).cast("double").alias(f"{c}__min"),
            F.max(c).cast("double").alias(f"{c}__max"),
            F.avg(c).cast("double").alias(f"{c}__avg"),
            F.sum((F.col(c) <= 0).cast("long")).alias(f"{c}__nonpos"),
            F.sum((F.col(c) < 0).cast("long")).alias(f"{c}__neg"),
        ]
    r = bronze(t).agg(*aggs).first().asDict()
    for c in cols:
        num_rows.append((t, c, r[f"{c}__min"], r[f"{c}__max"], r[f"{c}__avg"], r[f"{c}__nonpos"], r[f"{c}__neg"]))

display(spark.createDataFrame(
    num_rows, "table STRING, column STRING, min DOUBLE, max DOUBLE, avg DOUBLE, non_positive LONG, negative LONG"
))

### Discount and tax distribution

Both columns are rates stored as `DECIMAL(_, 2)`. The full distinct-value histogram shows the exact domain — if it is a small set of evenly populated steps, any value outside it is a defect, not a rare business case.

In [0]:
display(bronze("lineitem").groupBy("l_discount").count().orderBy("l_discount"))

In [0]:
display(bronze("lineitem").groupBy("l_tax").count().orderBy("l_tax"))

### Extended price consistency (informational)

In TPC-H, `l_extendedprice = l_quantity * p_retailprice`. This is not a silver rule (silver keeps the price as recorded on the line), but it tells us whether the monetary columns can be trusted as the input for net revenue.

In [0]:
ext_check = (
    bronze("lineitem").select("l_partkey", "l_quantity", "l_extendedprice")
    .join(bronze("part").select(F.col("p_partkey").alias("l_partkey"), "p_retailprice"), "l_partkey")
    .agg(
        F.count("*").alias("lines"),
        F.sum((F.abs(F.col("l_extendedprice") - F.col("l_quantity") * F.col("p_retailprice")) > 0.01).cast("long"))
         .alias("mismatches"),
    )
)
display(ext_check)

## 6. Categorical domains

The allowed-value lists in `02_silver` are taken from this output. Each list is small and closed; a new value appearing later should be quarantined and reviewed, not silently accepted.

In [0]:
CATEGORICAL_COLUMNS = [
    ("customer", "c_mktsegment"),
    ("orders",   "o_orderstatus"),
    ("orders",   "o_orderpriority"),
    ("lineitem", "l_returnflag"),
    ("lineitem", "l_linestatus"),
    ("lineitem", "l_shipmode"),
    ("lineitem", "l_shipinstruct"),
    ("part",     "p_mfgr"),
    ("part",     "p_container"),
    ("region",   "r_name"),
]

cat_df = None
for t, c in CATEGORICAL_COLUMNS:
    part_df = (bronze(t).groupBy(F.col(c).alias("value")).count()
               .select(F.lit(t).alias("table"), F.lit(c).alias("column"), "value", "count"))
    cat_df = part_df if cat_df is None else cat_df.unionByName(part_df)

display(cat_df.orderBy("table", "column", "value"))

## 7. Date ranges

In [0]:
DATE_COLUMNS = [
    ("orders",   "o_orderdate"),
    ("lineitem", "l_shipdate"),
    ("lineitem", "l_commitdate"),
    ("lineitem", "l_receiptdate"),
]
date_rows = []
for t, c in DATE_COLUMNS:
    r = bronze(t).agg(F.min(c).alias("mn"), F.max(c).alias("mx"), F.countDistinct(c).alias("n")).first()
    date_rows.append((t, c, r["mn"], r["mx"], r["n"]))
display(spark.createDataFrame(date_rows, "table STRING, column STRING, min DATE, max DATE, distinct_days LONG"))

## 8. Derived DQ bounds

The cell below computes the observed domain of every column that gets a range or allowed-value rule in silver and checks it against the bounds hard-coded in `02_silver`.
If the assertion fails, the source has changed and the bounds must be revisited — do not just widen them.

**How the bounds were chosen**

| Rule | Bound | Justification |
|---|---|---|
| `l_discount` | `0.00 ≤ x ≤ 0.10` | Observed min/max are exactly 0.00 and 0.10, every 0.01 step in between is populated with a near-uniform count, and this matches the TPC-H spec (clause 4.2.3: *random value [0.00 .. 0.10]*). Discount is a rate, so a value above 0.10 or below 0 in this dataset is a load defect, not a real deal. |
| `l_tax` | `0.00 ≤ x ≤ 0.08` | Same reasoning: observed 0.00–0.08 in uniform 0.01 steps, matches spec (*random value [0.00 .. 0.08]*). |
| `l_quantity`, `l_extendedprice`, `p_retailprice` | `> 0` | Finance rule. Profiling shows no zero/negative values; a zero-quantity or zero-price line would distort revenue and AOV. |
| `ps_supplycost` | `> 0` | Same; a free supply cost is not a valid business state. |
| `ps_availqty` | `≥ 0` | Stock can be zero but never negative. |
| `o_totalprice` | `> 0` | Every order has at least one positive line. |
| `c_acctbal`, `s_acctbal` | *no rule* | Negative balances are valid (spec range −999.99 … 9 999.99) and observed in the data. |
| Categorical lists | values from section 6 | Closed, small domains; any new value is quarantined for review. |

We pin bounds to the observed domain (which equals the spec domain) rather than to a looser "business" range such as `[0, 1]`, because this is generated data with a known domain — anything outside it means the pipeline broke.

In [0]:
# Must stay in sync with the constants in 02_silver
DISCOUNT_MIN, DISCOUNT_MAX = 0.00, 0.10
TAX_MIN, TAX_MAX = 0.00, 0.08
ALLOWED = {
    ("customer", "c_mktsegment"):  {"AUTOMOBILE", "BUILDING", "FURNITURE", "HOUSEHOLD", "MACHINERY"},
    ("orders",   "o_orderstatus"): {"F", "O", "P"},
    ("lineitem", "l_returnflag"):  {"A", "N", "R"},
    ("lineitem", "l_linestatus"):  {"F", "O"},
}

li = bronze("lineitem").agg(
    F.min("l_discount").cast("double").alias("dmin"), F.max("l_discount").cast("double").alias("dmax"),
    F.min("l_tax").cast("double").alias("tmin"),      F.max("l_tax").cast("double").alias("tmax"),
).first()

print(f"l_discount observed [{li.dmin:.2f}, {li.dmax:.2f}]  rule [{DISCOUNT_MIN:.2f}, {DISCOUNT_MAX:.2f}]")
print(f"l_tax      observed [{li.tmin:.2f}, {li.tmax:.2f}]  rule [{TAX_MIN:.2f}, {TAX_MAX:.2f}]")

problems = []
if not (DISCOUNT_MIN <= li.dmin and li.dmax <= DISCOUNT_MAX):
    problems.append("l_discount outside bounds")
if not (TAX_MIN <= li.tmin and li.tmax <= TAX_MAX):
    problems.append("l_tax outside bounds")

for (t, c), allowed in ALLOWED.items():
    observed = {r[0] for r in bronze(t).select(c).distinct().collect()}
    unexpected = observed - allowed
    print(f"{t + '.' + c:<24s} observed {sorted(map(str, observed))}")
    if unexpected:
        problems.append(f"{t}.{c} has unexpected values {unexpected}")

assert not problems, "Source no longer matches silver DQ rules: " + "; ".join(problems)
print("\nAll silver DQ bounds are consistent with the source.")